In [2]:
import pandas as pd
import os
import datetime

# 1. กำหนด Mapping สำหรับ location_id
location_mapping = {
    "57T": 1,  # เชียงราย
    "58T": 2,  # แม่ฮ่องสอน
    "82T": 3,  # หนองคาย
    "83T": 4,  # อุบลราชธานี
    "86T": 5,  # พิษณุโลก
    "79T": 6,  # กาญจนบุรี
    "05T": 7,  # กรุงเทพมหานคร
    "87T": 8,  # ตราด
    "42T": 9,  # สุราษฎร์ธานี
    "62T": 10  # นราธิวาส
}

start_date = '2025-03-10'
end_date = '2025-03-15'
fetched_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')

# 2. เตรียมส่วนหัวของ SQL
sql_header = "INSERT INTO `pm_actual` (`location_id`, `date`, `pm`, `temp`, `dew_point`, `humidity`, `pressure`, `wind_speed`, `precipitation`, `wind_direction`, `fetched_at`)\nVALUES"
values_list = []
current_id = 1

# 3. วนลูปอ่านแต่ละไฟล์และดึงข้อมูล
for file_prefix, loc_id in location_mapping.items():
    file_name = f"{file_prefix}.csv"
    
    if not os.path.exists(file_name):
        print(f"⚠️ หาไฟล์ {file_name} ไม่พบ ข้ามไฟล์นี้...")
        continue
        
    # อ่านไฟล์ CSV
    df = pd.read_csv(file_name)
    
    # กรองเฉพาะวันที่ต้องการ
    mask = (df['date'] >= start_date) & (df['date'] <= end_date)
    filtered_df = df[mask]
    
    # นำข้อมูลมาจัดรูปแบบ
    for _, row in filtered_df.iterrows():
        date = row['date']
        pm = row['pm25']
        temp = row['temperature_2m_mean']
        dew_point = row['dew_point_2m_mean']
        humidity = row['relative_humidity_2m_mean']
        pressure = row['surface_pressure_mean']
        wind_speed = row['wind_speed_10m_mean']
        precipitation = row['precipitation_sum']
        wind_direction = row['wind_direction_10m_dominant']
        
        # จัดการกรณีค่า pm เป็นค่าว่าง (NaN) ให้เป็น NULL ใน SQL
        pm_val = f"{pm}" if pd.notna(pm) else "NULL"
        
        value_str = f"({loc_id}, '{date}', {pm_val}, {temp}, {dew_point}, {humidity}, {pressure}, {wind_speed}, {precipitation}, {wind_direction}, '{fetched_at}')"
        values_list.append(value_str)
        current_id += 1

# 4. เขียนลงไฟล์
if values_list:
    final_sql = f"{sql_header}\n" + ",\n".join(values_list) + ";"
    
    with open('insert_pm_actual.sql', 'w', encoding='utf-8') as f:
        f.write(final_sql)
    print("✅ สร้างไฟล์ insert_pm_actual.sql เรียบร้อยแล้ว!")
else:
    print("❌ ไม่พบข้อมูลในช่วงวันที่ระบุในไฟล์ใดๆ เลย")

✅ สร้างไฟล์ insert_pm_actual.sql เรียบร้อยแล้ว!
